In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import h3
import os
from pathlib import Path
import glob

Una vez descargados los archivos y puestos en las carpetas de su clase, se junta todo en un df que contiene la ubicación geografica y la etiqueta asignada a cada negocio:

    Ejemplo, si el archivo se puso en la carpeta de bajos ingresos, el modelo aceptara que ese negocio se asocia a bajos ingresos

In [3]:
base_dir = os.getcwd()+r"\DATA\overpass"

carpetas = [
    elemento for elemento in os.listdir(base_dir ) 
    if os.path.isdir(os.path.join(base_dir , elemento))
    and len(os.listdir(os.path.join(base_dir, elemento))) > 0
    and elemento !="__pycache__"
]

In [6]:
diccionario={}
dfs = []
i = 1

for clase in carpetas:
    ruta = base_dir / clase
    archivos_csv = glob.glob(str(ruta / "*.csv"))
    diccionario[i]=clase
    df_temp = pd.concat([pd.read_csv(f, encoding='latin-1') for f in archivos_csv], ignore_index=True)
    df_temp["clase"] = i
    i += 1
    dfs.append(df_temp)
    
df_final = pd.concat(dfs, ignore_index=True)
df_final = df_final[['Longitud','Latitud','clase']]
df_final.dropna(inplace=True)

df_final.to_csv(base_dir / "Coordenadas_Negocios_Con_Clase_Asociada.csv", index=False)

In [7]:
df_final
diccionario

{1: 'C_AltoPoderAdquisitivo',
 2: 'C_Atraccion_Generacion_Z',
 3: 'C_Trafico_y_Consumo_Impulsivo',
 4: 'C_Universidades',
 5: 'C_Zonas_Residenciales_Delivery'}

In [12]:
RESOLUCION=9

In [ ]:
def obtener_hex_id(lat, lon, res):
    try:
        return h3.latlng_to_cell(lat, lon, res) 
    except AttributeError:
        return h3.geo_to_h3(lat, lon, res) 

In [20]:
df_final['hex_id'] = df_final.apply(
    lambda row: obtener_hex_id(row['Latitud'], row['Longitud'], RESOLUCION), 
    axis=1
)


matriz = pd.crosstab(df_final['hex_id'], df_final['clase']).fillna(0)


matriz.columns = [diccionario.get(col, f'clase_{col}') for col in matriz.columns]

In [22]:
matriz

,C_AltoPoderAdquisitivo,C_Atraccion_Generacion_Z,C_Trafico_y_Consumo_Impulsivo,C_Universidades,C_Zonas_Residenciales_Delivery
hex_id,,,,,
89498649a8bffff,1,0,0,0,0
8949958082fffff,1,0,0,0,0
894995808a7ffff,0,0,1,0,0
89499584017ffff,0,2,0,0,0
89499584047ffff,0,0,1,1,0
...,...,...,...,...,...
894995bb597ffff,0,1,0,0,0
894995bb5a3ffff,0,0,0,1,0
894995bb5b7ffff,1,0,0,0,0


In [24]:
def dibujar_poligono_hex(hex_id):
    try:
        boundary = h3.cell_to_boundary(hex_id) # Versión H3 4.0+
    except AttributeError:
        boundary = h3.h3_to_geo_boundary(hex_id) # Versión H3 3.x
        
    return Polygon([(lng, lat) for lat, lng in boundary])


matriz['geometry'] = matriz.index.map(dibujar_poligono_hex)


gdf_hexagonos = gpd.GeoDataFrame(matriz, geometry='geometry', crs="EPSG:4326")


nombre_shp = "Mercados_Hexagonales_H3.shp"
gdf_hexagonos.to_file(nombre_shp, driver="ESRI Shapefile")

print(f"Exportación exitosa a {nombre_shp}.")

Exportación exitosa a Mercados_Hexagonales_H3.shp.


C:\Users\xboxn\AppData\Local\Temp\ipykernel_28632\450827049.py:17: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_hexagonos.to_file(nombre_shp, driver="ESRI Shapefile")
C:\Users\xboxn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'C_AltoPoderAdquisitivo' to 'C_AltoPode'
  ogr_write(
C:\Users\xboxn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'C_Atraccion_Generacion_Z' to 'C_Atraccio'
  ogr_write(
C:\Users\xboxn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'C_Trafico_y_Consumo_Impulsivo' to 'C_